# 파인튜닝 워크샵 (RunPod)

**LLM SFT · Tool Calling · Embedding FT — 올인원 실습**

| Part | 주제 | 모델 | 데이터 | 평가 |
|------|------|------|--------|------|
| **Part 1** | LLM SFT | **Qwen3-4B-Base** (QLoRA) | KoAlpaca-v1.1a | ROUGE-L (kiwipiepy) |
| **Part 2** | Tool Calling FT | **Qwen3-4B-Instruct-2507** | glaive-function-calling-v2 | Tool Call 정확도 |
| **Part 3** | Embedding FT | BAAI/bge-m3 | 직접 합성 (RunPod) | Spearman · Matryoshka |

<br>

> **왜 Base/Instruct를 나눠 쓰는가?** SFT는 '일반 지시 따르기'를 Base에 처음 주입하는 과정 → `Qwen3-4B-Base`에서 출발.  
> Tool Calling은 이미 지시 튜닝된 모델 위에 '함수 호출 포맷'만 얹는 작업 → `Qwen3-4B-Instruct-2507`에서 출발.  

> **환경**: RunPod RTX 4090 (24GB) · Docker `unsloth/unsloth:latest` · Pod 환경변수 `JUPYTER_PASSWORD`

---

## 0. 환경 설정

> 💡 **이 섹션에서 하는 일**
> 1. GPU/VRAM 확인 (RunPod RTX 4090 = 24GB 기준)
> 2. 추가 패키지 설치 (Docker 이미지에 없는 것만)
> 3. HuggingFace 인증 — 모델 다운로드·LoRA 업로드에 필요

In [ ]:
import torch, os, json, re, random, warnings
warnings.filterwarnings('ignore')

# GPU 상태 점검 (bf16: T4=❌, A100/4090=✅)
print(f"GPU: {torch.cuda.get_device_name(0)}")
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"VRAM: {vram:.1f} GB | BF16: {torch.cuda.is_bf16_supported()}")

# RunPod 영구 볼륨 (/workspace는 Pod 재시작 후에도 유지)
WORK_DIR = "/workspace/finetuning"
os.makedirs(WORK_DIR, exist_ok=True)

In [ ]:
# Unsloth Docker 이미지에 없는 패키지 추가 설치
!pip install -q "sentence-transformers>=3.0" rouge-score kiwipiepy hf_transfer

In [ ]:
import unsloth  # Docker 이미지에 사전 설치됨

# HuggingFace 인증: 환경변수 우선, 없으면 getpass로 입력
import os
from huggingface_hub import login
from getpass import getpass

hf_token = os.getenv("HUGGINGFACE_TOKEN") or os.getenv("HF_TOKEN")
if not hf_token:
    hf_token = getpass("🔑 HuggingFace Token 입력 (https://hf.co/settings/tokens): ")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=True)
print("✅ HF 인증 완료")

---
# Part 1: LLM SFT — `Qwen3-4B-Base` 기반

**목표**: pretrain만 된 Base 모델에 **지시 따르기(instruction-following)** 를 처음 주입

> 💡 **SFT (Supervised Fine-Tuning) 한 줄 정의**
> "질문 → 정답" 쌍을 모델에 보여주며 정답 쪽 토큰의 loss만 학습시켜, 모델이 **지시를 따르는 대화체**를 익히도록 하는 과정. Pretrain된 모델은 "다음 단어 예측"만 할 줄 알고, SFT가 "질문에 답하는 법"을 가르칩니다.

**핵심 기법 3가지**
1. **ChatML 포맷**: `<|im_start|>role ... <|im_end|>` 구조로 대화를 직렬화
2. **`train_on_responses_only`**: user/assistant 마커로 어시스턴트 턴의 loss만 계산
3. **한국어 ROUGE (kiwipiepy)**: 영문 토크나이저는 한국어 어미/조사 때문에 점수 0 수렴 → 형태소 분리 필수

### 1.1 Chat Template (`qwen3-instruct`) 구조

SFT 학습 데이터는 결국 **하나의 긴 문자열**로 직렬화됩니다. Qwen3의 ChatML 포맷은 다음과 같은 모양입니다.

```text
<|im_start|>system
당신은 도움이 되는 AI 어시스턴트입니다.<|im_end|>
<|im_start|>user
한국의 수도는?<|im_end|>
<|im_start|>assistant
대한민국의 수도는 서울입니다.<|im_end|>
```

- `<|im_start|>`, `<|im_end|>`는 모델이 턴의 시작/끝을 인식하는 **special token**
- `system`·`user`·`assistant`는 역할 태그

#### `train_on_responses_only` 작동 원리

학습 시 모든 토큰을 loss에 반영하면 모델이 **user 질문까지 외우려** 하게 됩니다. `train_on_responses_only`는 다음 마커 사이 구간만 loss 대상으로 표시:

- **instruction_part**: `<|im_start|>user\n` (이 뒤 구간은 **마스킹** → loss 무시)
- **response_part**: `<|im_start|>assistant\n` (이 뒤 구간만 **학습**)

→ 모델은 "user 질문이 주어졌을 때 assistant가 할 답"만 학습.

#### 💡 Base 모델에 Instruct 템플릿을 이식해도 되는가?

`Qwen3-Base`는 `Qwen3-Instruct`와 **동일한 토크나이저**(특수 토큰 `<|im_start|>` 등 포함)로 pretrain됩니다. 즉, chat template의 스페셜 토큰은 이미 Base의 임베딩 공간에 존재합니다. SFT는 "이 토큰들을 **대화 구조로 사용하는 법**"을 가르치는 과정입니다.

#### ChatML vs Alpaca (참고)

|              | ChatML (Qwen3)                                      | Alpaca                                         |
|--------------|-----------------------------------------------------|------------------------------------------------|
| 형태         | `<im_start>user...assistant...`         | `### Instruction:... ### Response:`            |
| 다중턴       | 자연스러움                                           | 어색함 (append 필요)                            |
| Tool 지원    | `tool_call` 역할 내장                                | 별도 규약 필요                                  |

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset

# Part 1: Base 모델 (pretrain만 된 상태)
SFT_MODEL = "unsloth/Qwen3-4B-Base"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL,
    max_seq_length=2048,       # ✏️ 시퀀스 길이
    load_in_4bit=True,         # QLoRA 4bit 양자화
    load_in_8bit=False,
    full_finetuning=False,     # LoRA 어댑터만 학습
    token=os.environ.get("HF_TOKEN"),
)
# Base 모델에 Instruct 템플릿 이식
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

# LoRA 어댑터 주입: 원본 가중치는 동결, A·B 행렬만 학습
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                      # ✏️ LoRA 랭크
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,            # Unsloth는 0일 때 최적
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ {SFT_MODEL} + LoRA r=16 ({trainable/1e6:.1f}M / {total/1e6:.1f}M)")

In [ ]:
# KoAlpaca → ChatML 변환 + 품질 필터링
raw = load_dataset("beomi/KoAlpaca-v1.1a", split="train")
print(f"KoAlpaca: {len(raw)}개")

SYSTEM_PROMPT = "당신은 도움이 되는 AI 어시스턴트입니다."

def to_chatml(sample):
    user = sample["instruction"]
    if sample.get("input", "").strip():
        user += f"\n\n{sample['input']}"
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
        {"role": "assistant", "content": sample["output"]},
    ]}

# 필터링: output 20~1500자, instruction 중복 제거
seen = set()
clean = []
for s in raw:
    out = s.get("output", "")
    inst = s.get("instruction", "")
    if not out.strip() or len(out) < 20 or len(out) > 1500:
        continue
    if inst in seen:
        continue
    seen.add(inst)
    clean.append(to_chatml(s))
    if len(clean) >= 3000:              # ✏️ 데이터 크기
        break

from datasets import Dataset
sft_data = Dataset.from_list(clean)
split = sft_data.train_test_split(test_size=0.05, seed=42)
print(f"✅ 학습: {len(split['train'])}, 평가: {len(split['test'])}")

In [ ]:
# Before 기준선: 학습 전 모델 응답 (Base 모델은 아직 대화 해석 못 함 → 이상해도 정상)
FastLanguageModel.for_inference(model)

EVAL_QS = [
    {"q":"한국의 수도는?", "ref":"대한민국의 수도는 서울입니다."},
    {"q":"인공지능의 윤리적 문제는?", "ref":"AI 윤리 문제로는 편향성, 개인정보 침해, 일자리 대체 등이 있습니다."},
]

def gen(q, max_tok=256):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": q},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_tok,
            temperature=0.7,           # ✏️ 낮으면 보수적, 높으면 창의적
            top_p=0.8,                 # ✏️ Qwen3 권장값
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[tokenizer.eos_token_id, im_end_id],
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

before = [gen(e["q"]) for e in EVAL_QS]
for i, e in enumerate(EVAL_QS):
    print(f"Q: {e['q']}\nA: {before[i][:120]}...\n")

### 1.2 한국어 ROUGE 평가 준비 (kiwipiepy)

**문제**: `rouge_score` 기본 토크나이저는 `\w+` 정규식 기반. 한국어는 "수도는" / "수도입니다"가 완전히 다른 어절로 인식되어 n-gram 매칭이 거의 0.

**해결**: `kiwipiepy` 형태소 분석기로 조사·어미를 분리한 뒤 ROUGE 계산.

```text
"한국의 수도는 서울입니다"
  ↓ 공백 분리:  ['한국의', '수도는', '서울입니다']    ← 어절 덩어리, 재사용 거의 불가
  ↓ 형태소 분리: ['한국', '의', '수도', '는', '서울', '이', 'ㅂ니다']  ← 재사용 가능한 단위
```

In [ ]:
from kiwipiepy import Kiwi
from rouge_score import rouge_scorer

class KoreanTokenizer:
    """kiwipiepy 형태소 기반 토크나이저 — 한국어 ROUGE 평가용"""
    def __init__(self):
        self._kiwi = Kiwi()
    def tokenize(self, text):
        return [t.form for t in self._kiwi.tokenize(text)]

ko_tok = KoreanTokenizer()
ko_scorer = rouge_scorer.RougeScorer(['rougeL'], tokenizer=ko_tok)

# 공백 분리 vs 형태소 분리 비교
sample = "대한민국의 수도는 서울입니다."
print(f"원문:          {sample}")
print(f"공백 분리:     {sample.split()}")
print(f"형태소 분리:   {ko_tok.tokenize(sample)}")

# Smoke test
ref = "한국의 수도는 서울입니다"
pred = "한국 수도 서울"
score = ko_scorer.score(ref, pred)['rougeL'].fmeasure
print(f"\n📐 Smoke test — ref='{ref}' vs pred='{pred}'")
print(f"   rougeL = {score:.3f} (형태소 기반이므로 > 0)")

In [ ]:
# pad/eos 토큰 확인
print("pad_token:", repr(tokenizer.pad_token), tokenizer.pad_token_id)
print("eos_token:", repr(tokenizer.eos_token), tokenizer.eos_token_id)

In [ ]:
# SFT 학습 (train_on_responses_only)
from trl import SFTTrainer, SFTConfig

def to_text(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
            for m in batch["messages"]
        ]
    }

ds = split.map(to_text, batched=True, remove_columns=split["train"].column_names)

# ChatML 변환 확인
print(ds["train"][0]["text"][:500])

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    args=SFTConfig(
        output_dir=f"{WORK_DIR}/sft",
        num_train_epochs=2,                     # ✏️ epoch 수
        per_device_train_batch_size=8,          # ✏️ OOM 나면 줄이기
        gradient_accumulation_steps=4,          # effective batch = 8 × 4 = 32
        learning_rate=2e-4,                     # ✏️ LoRA 표준값
        warmup_steps=5,
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        bf16=True,
        max_seq_length=2048,
        logging_steps=10,
        report_to="none",
        seed=42,
        dataset_text_field="text",
    ),
)
# user 질문은 마스킹, assistant 응답만 학습
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
# 학습 실행 (4090 기준 ~10-15분)
print("🚀 SFT 학습 시작...")
trainer_stats = trainer.train()

print(f"\n✅ 학습 완료")
print(f"  - 총 소요시간: {trainer_stats.metrics['train_runtime']:.1f}초")
print(f"  - 초당 샘플: {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"  - 최종 train loss: {trainer_stats.metrics['train_loss']:.4f}")

# 검증셋 평가
eval_metrics = trainer.evaluate()
print(f"  - eval loss: {eval_metrics['eval_loss']:.4f}")

# 피크 VRAM
import torch
peak_mem = torch.cuda.max_memory_reserved() / 1024**3
print(f"  - Peak GPU memory: {peak_mem:.2f} GB")

In [ ]:
# ROUGE Before/After 비교
FastLanguageModel.for_inference(model)
after = [gen(e["q"]) for e in EVAL_QS]

print("📊 ROUGE-L Before/After (한국어 형태소 토큰화):")
print(f"  {'질문':<22} {'Before':>8}  {'After':>8}  {'Δ':>8}")
print("  " + "-"*52)
for i, e in enumerate(EVAL_QS):
    b = ko_scorer.score(e["ref"], before[i])['rougeL'].fmeasure
    a = ko_scorer.score(e["ref"], after[i])['rougeL'].fmeasure
    q = e["q"][:20]
    print(f"  {q:<22} {b:>8.3f}  {a:>8.3f}  {a-b:>+8.3f}")

# ⚠️ ROUGE는 어휘 중복 기반 → 실무에선 BERTScore·LLM-as-judge 병용 권장

# LoRA 어댑터 로컬 저장 (수십~수백 MB)
model.save_pretrained(f"{WORK_DIR}/sft_lora")
tokenizer.save_pretrained(f"{WORK_DIR}/sft_lora")
print(f"\n✅ SFT LoRA 저장 완료")

# (선택) HuggingFace Hub 업로드
HF_USERNAME = "redwiggler"             # ✏️ 본인 HF 계정명
REPO_NAME   = "qwen3-4b-sft-lora"      # ✏️ 리포 이름
HUB_REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"
HF_TOKEN    = os.environ.get("HF_TOKEN")

model.push_to_hub(HUB_REPO_ID, token=HF_TOKEN, private=True)
tokenizer.push_to_hub(HUB_REPO_ID, token=HF_TOKEN, private=True)
print(f"✅ LoRA 업로드 완료: https://huggingface.co/{HUB_REPO_ID}")

---
# Part 2: Tool Calling FT — `Qwen3-4B-Instruct-2507` 기반

**목표**: 이미 지시 튜닝된 Instruct 모델 위에 **함수 호출 포맷**만 얹기

> 💡 **Tool Calling 한 줄 정의**
>
> LLM이 사용자 질문을 보고 "직접 답할지" vs "외부 함수(API·DB·검색)를 호출할지"를 판단해서, 호출이 필요하면 **JSON 형태의 함수 호출 블록을 출력**하도록 만드는 기능. 예: "서울 날씨 알려줘" → `{"name": "get_weather", "arguments": {"city": "서울"}}`. 이 JSON을 받아서 실제 함수를 실행하는 건 앱 코드의 몫.

### 왜 Part 1(Base)과 다른 모델을 쓰는가?

| | Part 1 SFT | Part 2 Tool Calling |
|---|---|---|
| **출발 모델** | Qwen3-4B-**Base** | Qwen3-4B-**Instruct-2507** |
| **이미 학습된 것** | 언어 모델링만 | 언어 + 지시 따르기 + 대화 형식 |
| **추가 학습할 것** | 지시 응답 + 대화 포맷 | `<tool_call>` JSON 구조 |
| **데이터 포맷** | ChatML (일반 대화) | ChatML + tool/tool_call 역할 |
| **LoRA r** | 16 (범용) | **32** (구조화 출력에 더 많은 용량) |
| **Temperature** | 0.7 (창의성) | **0.1** (JSON 정합성) |

### 핵심 기법

1. **glaive 데이터 포맷 변환**: `<functioncall>...` → Qwen3 `<tool_call>...</tool_call>`
2. **균형 샘플링 (50:50)**: Tool을 호출해야 하는 질의 ↔ 일반 대화 → 과호출·미호출 방지
3. **VRAM 명시 해제**: Part 1 모델을 `del` + `torch.cuda.empty_cache()`로 비우고 새 모델 로드

### 2.1 Tool Calling ChatML 구조

Qwen3는 함수 호출을 위해 **4가지 역할**을 지원합니다:

```text
<|im_start|>system
[사용 가능한 함수 목록과 스키마가 여기 주입됨]<|im_end|>
<|im_start|>user
서울 날씨 알려줘<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "get_weather", "arguments": {"city": "서울"}}
</tool_call><|im_end|>
<|im_start|>tool
{"temperature": 15, "condition": "맑음"}<|im_end|>
<|im_start|>assistant
서울은 현재 15도, 맑은 날씨입니다.<|im_end|>
```

- **`<tool_call>...</tool_call>`**: 모델이 호출을 요청하는 JSON 블록
- **`tool` 역할**: 실제 함수 실행 결과를 다시 모델에 전달

### glaive `<functioncall>` → Qwen3 `<tool_call>` 변환

```text
원본 glaive:   ASSISTANT: <functioncall> {"name":"...", "arguments":"..."}
  ↓ 정규식 추출 + JSON 파싱 + arguments를 dict로 변환
Qwen3:        <tool_call>\n{"name":"...", "arguments":{...}}\n</tool_call>
```

### 💡 왜 추론 시 `temperature=0.1`인가?

- `temperature=0.0`(greedy)는 **반복/loop 위험** (같은 토큰 무한 생성)
- `temperature=0.1`은 JSON **구조(틀)는 거의 deterministic**, 동시에 arguments 안의 문자열 필드에 **약간의 다양성** 허용
- 실무 tool-call 엔드포인트도 보통 0.1 기본값

In [ ]:
# VRAM 해제: Part 1 모델 제거 (del + gc + empty_cache 3단 세트)
import gc

before = torch.cuda.memory_allocated() / 1024**3
try:
    del model, trainer, tokenizer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
after = torch.cuda.memory_allocated() / 1024**3
print(f"🧹 VRAM: {before:.2f} GB → {after:.2f} GB "
      f"(reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB)")

# Part 2: Instruct 모델 로드 (Tool Calling의 출발점)
TC_MODEL = "unsloth/Qwen3-4B-Instruct-2507"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=TC_MODEL, max_seq_length=2048,
    load_in_4bit=True, load_in_8bit=False, full_finetuning=False,
    token=os.environ.get("HF_TOKEN"),
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # Tool Calling은 JSON 구조 학습이 필요 → r=16보다 큰 capacity
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print(f"✅ {TC_MODEL} + LoRA r=32 (Tool Calling)")
print(f"   VRAM after load: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
# glaive <functioncall> 파싱 헬퍼
# 처리 이슈: (1) arguments가 single quote로 감싼 문자열, (2) 중첩 JSON 구조
def extract_functioncall(text):
    """glaive의 <functioncall> {...} 블록에서 name/arguments 추출."""
    idx = text.find("<functioncall>")
    if idx == -1:
        return None

    start = text.find("{", idx)
    if start == -1:
        return None

    # 괄호 균형 파서: 문자열 내부의 {}는 무시하고 중첩 JSON 안전 처리
    depth = 0
    in_str = False
    str_char = None
    end = -1

    for i in range(start, len(text)):
        c = text[i]
        if in_str:
            if c == "\\":
                continue
            if c == str_char:
                in_str = False
        else:
            if c in ('"', "'"):
                in_str = True
                str_char = c
            elif c == "{":
                depth += 1
            elif c == "}":
                depth -= 1
                if depth == 0:
                    end = i
                    break

    if end == -1:
        return None

    raw = text[start:end+1]

    # single quote로 감싼 arguments를 normal JSON으로 변환
    fixed = re.sub(
        r'"arguments":\s*\'(\{.*?\})\'',
        lambda m: '"arguments": ' + m.group(1),
        raw,
        flags=re.DOTALL,
    )

    try:
        d = json.loads(fixed)
    except json.JSONDecodeError:
        return None

    name = d.get("name", "")
    args = d.get("arguments", {})

    # arguments가 여전히 문자열이면 한 번 더 파싱
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            pass

    return {"name": name, "arguments": args}

In [ ]:
# glaive → Qwen3 ChatML+Tool 포맷 변환
import json, re
from tqdm import tqdm

raw_glaive = load_dataset("glaiveai/glaive-function-calling-v2", split="train")
print(f"glaive: {len(raw_glaive)}개")

def parse_glaive(sample):
    chat = sample.get("chat", "")
    system = sample.get("system", "")
    if not chat.strip():
        return None

    messages = []
    if system.strip():
        system_text = system.strip()
        if system_text.startswith("SYSTEM:"):
            system_text = system_text[len("SYSTEM:"):].strip()
        messages.append({"role": "system", "content": system_text})


    # 역할 마커로 분할
    parts = re.split(r'(USER:|ASSISTANT:|FUNCTION RESPONSE:)', chat)
    role = None
    for p in parts:
        p = p.strip()
        if p == "USER:":
            role = "user"
        elif p == "ASSISTANT:":
            role = "assistant"
        elif p == "FUNCTION RESPONSE:":
            role = "tool"
        elif role and p:
            content = p

            # glaive 포맷 → Qwen3 <tool_call> 포맷 변환
            if role == "assistant" and "<functioncall>" in content:
                fc = extract_functioncall(content)
                if fc is None:
                    return None
                preamble = content.split("<functioncall>")[0].strip()
                tc = f"<tool_call>\n{json.dumps(fc, ensure_ascii=False)}\n</tool_call>"
                content = f"{preamble}\n{tc}" if preamble else tc

            elif role == "tool":
                # 함수 응답은 유효한 JSON이어야 함
                try:
                    json.loads(content)
                except json.JSONDecodeError:
                    return None

            messages.append({"role": role, "content": content})

    # 품질 체크: 2턴 이상, 마지막은 assistant, tool 앞에 tool_call 존재
    if len(messages) < 2 or messages[-1]["role"] != "assistant":
        return None
    for i, m in enumerate(messages):
        if m["role"] == "tool":
            if i == 0 or messages[i-1]["role"] != "assistant":
                return None
            if "<tool_call>" not in messages[i-1]["content"]:
                return None

    return {"messages": messages}

converted = []
for s in tqdm(raw_glaive, desc="변환 중"):
    r = parse_glaive(s)
    if r:
        converted.append(r)
print(f"변환 성공: {len(converted)}/{len(raw_glaive)} ({len(converted)/len(raw_glaive):.1%})")

In [ ]:
# 균형 샘플링: Tool 호출 대화 vs 일반 대화 50:50 (over-calling 편향 방지)
tool_s, gen_s = [], []
for s in converted:
    is_tool = any(
        "<tool_call>" in m["content"]
        for m in s["messages"]
        if m["role"] == "assistant"
    )
    (tool_s if is_tool else gen_s).append(s)
print(f"Tool: {len(tool_s)}, General: {len(gen_s)}")

random.seed(42)
N = 1000                                    # ✏️ 각 그룹별 샘플 수
n_tool = min(N, len(tool_s))
n_gen  = min(N, len(gen_s))
if n_tool < N or n_gen < N:
    print(f"⚠️ 목표 {N}개 미달 → tool={n_tool}, general={n_gen}")

balanced = random.sample(tool_s, n_tool) + random.sample(gen_s, n_gen)
random.shuffle(balanced)

tool_data = Dataset.from_list(balanced)
print(f"✅ 균형 데이터: {len(tool_data)}개 (tool {n_tool} + general {n_gen})")

> tool calling 학습 시, tool 샘플만으로 파인튜닝하면 모델이 **모든 질문에 함수를 호출하려는 편향**(over-calling)이 생김. general 샘플을 섞어 "함수가 필요 없을 때는 바로 답한다"를 같이 학습.

In [ ]:
# ChatML 텍스트로 직렬화
def tc_to_text(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
            for m in batch["messages"]
        ]
    }

tc_ds = tool_data.map(tc_to_text, batched=True, remove_columns=tool_data.column_names)
tc_split = tc_ds.train_test_split(test_size=0.1, seed=42)
print(f"✅ Tool ChatML: train={len(tc_split['train'])}, eval={len(tc_split['test'])}")
print("샘플(앞 400자):"); print(tc_split["train"][0]["text"][:400])

# Tool Calling 학습 설정
trainer_tc = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=tc_split["train"],
    eval_dataset=tc_split["test"],
    args=SFTConfig(
        output_dir=f"{WORK_DIR}/tool_calling",
        num_train_epochs=1,
        per_device_train_batch_size=2,              # Tool 대화는 시퀀스가 길어 작게
        gradient_accumulation_steps=8,              # effective batch = 16
        learning_rate=2e-4, warmup_steps=10,
        lr_scheduler_type="cosine", optim="adamw_8bit",
        bf16=True, max_seq_length=2048,
        logging_steps=10,
        eval_strategy="steps", eval_steps=20,
        report_to="none", seed=42,
        dataset_text_field="text",
    ),
)
# assistant 턴(tool_call JSON 포함)만 학습
trainer_tc = train_on_responses_only(
    trainer_tc,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# 라벨 마스킹 검증 (labels == -100은 loss 제외)
sample = trainer_tc.train_dataset[0]
learned = [tid for tid, lab in zip(sample["input_ids"], sample["labels"]) if lab != -100]
print("\n📝 학습 대상 구간 미리보기:")
print(tokenizer.decode(learned)[:300], "...")

print("\n🚀 Tool Calling 학습 시작...")
tc_stats = trainer_tc.train()
print(f"✅ Tool Calling 완료 | Loss: {tc_stats.training_loss:.4f} | "
      f"Runtime: {tc_stats.metrics['train_runtime']:.1f}s | "
      f"Peak VRAM: {torch.cuda.max_memory_reserved()/1024**3:.2f} GB")

In [ ]:
# Part 2 평가: Tool Calling 정확도 4개 축(A/B/C/D) + 멀티턴 시뮬레이션
#   A. 명확한 함수 호출  B. 유사 표현 일반화  C. Over-calling 방지  D. Hallucination 방지
FastLanguageModel.for_inference(model)


# 함수 스키마 정의: 3가지 도메인 (OpenAI function calling 표준 포맷)
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "도시의 현재 날씨를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "도시명 (예: 서울, 도쿄)"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"],
                             "description": "온도 단위"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_restaurants",
            "description": "지역과 음식 종류로 식당을 검색합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location":    {"type": "string", "description": "검색 지역 (예: 강남, 홍대)"},
                    "cuisine":     {"type": "string", "description": "음식 종류 (예: 한식, 이탈리안)"},
                    "price_range": {"type": "string", "enum": ["저렴", "보통", "고급"]},
                },
                "required": ["location"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_expense_ratio",
            "description": "ETF의 총보수(연간 운용비용)를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "etf_name": {"type": "string", "description": "ETF 이름 (예: KODEX 200)"},
                },
                "required": ["etf_name"],
            },
        },
    },
]


# 종료 토큰: eos와 <|im_end|> 둘 다 등록
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
EOS_IDS = [tokenizer.eos_token_id, IM_END_ID]


def gen_tool(messages, deterministic=True, max_new_tokens=256):
    """메시지 → assistant 응답. deterministic=True면 greedy(평가 재현성)."""
    inputs = tokenizer.apply_chat_template(
        messages,
        tools=TOOLS,                      # 함수 스키마 주입
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=EOS_IDS,
    )
    if deterministic:
        gen_kwargs["do_sample"] = False
    else:
        gen_kwargs.update(do_sample=True, temperature=0.1, top_p=0.9)

    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)

    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )


def extract_tool_call(resp):
    """응답에서 <tool_call>...</tool_call> JSON 파싱."""
    m = re.search(r'<tool_call>\s*(.+?)\s*</tool_call>', resp, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(1))
    except json.JSONDecodeError:
        return None


# 단일턴 평가: (질문, 호출 기대 여부, 기대 함수명 or None)
tests = [
    # A. 명확한 함수 호출
    ("서울 날씨 알려줘",                      True,  "get_weather"),
    ("강남에서 이탈리안 식당 찾아줘",          True,  "search_restaurants"),
    ("KODEX 200 총보수가 얼마야?",            True,  "calculate_expense_ratio"),

    # B. 유사 표현 (의미 기반 선택)
    ("도쿄 기온이 어때?",                      True,  "get_weather"),
    ("홍대 근처 맛집 추천해줘",                True,  "search_restaurants"),
    ("TIGER 나스닥100 운용수수료는?",          True,  "calculate_expense_ratio"),

    # C. 호출하면 안 됨 (over-calling 진단)
    ("파이썬이 뭐야?",                        False, None),
    ("안녕하세요",                            False, None),
    ("날씨 앱 추천해줘",                      False, None),   # 키워드 매칭 아닌 의도 파악

    # D. 정의 밖 요청 (hallucination 진단)
    ("비트코인 가격 알려줘",                  False, None),
    ("영화 예매해줘",                         False, None),
]


test_axes = ["A"]*3 + ["B"]*3 + ["C"]*3 + ["D"]*2

axis_labels = {
    "A": "명확한 함수 호출 (함수 선택 정확도)",
    "B": "유사 표현 (의미 기반 선택)",
    "C": "호출하면 안 됨 (over-calling 방지)",
    "D": "정의 밖 요청 (hallucination 방지)",
}

results = {"A": [0, 0], "B": [0, 0], "C": [0, 0], "D": [0, 0]}

print("🤖 Tool Calling 평가 (3개 도구 × 4개 평가 축)")

current_axis = None
for (q, expect, expect_fn), axis in zip(tests, test_axes):
    if axis != current_axis:
        current_axis = axis
        print(f"\n── {axis}. {axis_labels[axis]} ──")

    resp = gen_tool([{"role": "user", "content": q}])
    has_call = "<tool_call>" in resp

    if expect:
        # 호출해야 + 올바른 함수
        tc = extract_tool_call(resp)
        called_fn = tc.get("name") if tc else None
        ok = has_call and called_fn == expect_fn
        detail = (f"호출={called_fn} (기대={expect_fn})"
                  if has_call else "호출 없음 (기대=있음)")
    else:
        # 호출 안 해야 정답
        ok = not has_call
        if has_call:
            tc = extract_tool_call(resp)
            called_name = tc.get("name") if tc else "파싱실패"
            detail = f"호출={called_name} (기대=없음)"
        else:
            detail = "호출 없음 ✓"

    results[axis][0] += ok
    results[axis][1] += 1
    mark = "✅" if ok else "❌"
    print(f"  {mark} {q:<28} → {detail}")


# 축별 정확도 요약
print("\n" + "═" * 60)
print("📊 평가 축별 정확도")
print("═" * 60)

total_correct, total_count = 0, 0
for axis in ["A", "B", "C", "D"]:
    c, t = results[axis]
    total_correct += c
    total_count   += t
    print(f"  {axis}. {axis_labels[axis]:<35}: {c}/{t} ({c/t:.0%})")

print(f"\n  🎯 전체 정확도: {total_correct}/{total_count} ({total_correct/total_count:.0%})")


# 멀티턴 시뮬레이션: tool_call → mock 실행 → 최종 답변 (agentic loop 1턴)
MOCK_RESULTS = {
    "get_weather": {
        "city": "서울", "temp": 15, "condition": "맑음", "humidity": 45,
    },
    "search_restaurants": {
        "results": [
            {"name": "라그릴리아",  "rating": 4.5, "price": "고급"},
            {"name": "파스타부띠끄", "rating": 4.3, "price": "보통"},
        ],
    },
    "calculate_expense_ratio": {
        "etf_name": "KODEX 200", "expense_ratio": 0.15, "unit": "%",
    },
}


def simulate_multiturn(q, mock_results):
    """User 질문 → tool_call → mock 실행 → 최종 답변까지 전체 사이클."""
    msgs = [{"role": "user", "content": q}]
    print(f"\n[1] 👤 User    : {q}")

    resp1 = gen_tool(msgs).strip()
    print(f"[2] 🤖 Assistant: {resp1}")

    tc = extract_tool_call(resp1)
    if tc is None:
        print("    (함수 호출 없이 바로 답변 종료)")
        return

    # Mock backend 실행
    fn_name = tc.get("name", "")
    tool_result = mock_results.get(fn_name, {"error": f"unknown tool: {fn_name}"})
    tool_result_str = json.dumps(tool_result, ensure_ascii=False)
    print(f"[3] 🔧 Tool     : {fn_name}("
          f"{json.dumps(tc.get('arguments', {}), ensure_ascii=False)})")
    print(f"    ↳ result : {tool_result_str}")

    # tool 결과를 대화에 추가하고 재호출 → 자연어 답변
    msgs.extend([
        {"role": "assistant", "content": resp1},
        {"role": "tool",      "content": tool_result_str},
    ])
    resp2 = gen_tool(msgs).strip()
    print(f"[4] 🤖 Assistant: {resp2}")


print("\n" + "═" * 60)
print("🔄 멀티턴 시뮬레이션 (3개 도구 각각 시연)")
print("═" * 60)

for q in [
    "서울 날씨 알려줘",
    "강남에서 이탈리안 식당 찾아줘",
    "KODEX 200 총보수가 얼마야?",
]:
    simulate_multiturn(q, MOCK_RESULTS)


# LoRA 저장
model.save_pretrained(f"{WORK_DIR}/tool_calling_lora")
tokenizer.save_pretrained(f"{WORK_DIR}/tool_calling_lora")
print(f"\n✅ Tool Calling LoRA 저장 완료: {WORK_DIR}/tool_calling_lora")


# HF Hub 업로드 (HF_TOKEN은 write 권한 필요)
HF_USERNAME = "redwiggler"
HUB_REPO_ID = f"{HF_USERNAME}/qwen3-4b-tool-calling-lora"

model.push_to_hub(HUB_REPO_ID, token=os.environ.get("HF_TOKEN"), private=True)
tokenizer.push_to_hub(HUB_REPO_ID, token=os.environ.get("HF_TOKEN"), private=True)
print(f"✅ Hub 업로드: https://huggingface.co/{HUB_REPO_ID}")

---
# Part 3: Embedding FT — BGE-M3

**목표**: 도메인 특화 검색 품질 향상 (일반 임베딩 모델은 ETF 같은 전문 도메인에서 범주 구별이 약함)

> 💡 **Embedding FT는 LLM FT와 무엇이 다른가?**
> - LLM FT(Part 1·2) = "답변을 생성하는 법" 학습 → 출력이 **텍스트**
> - Embedding FT(Part 3) = "의미가 비슷한 문장을 가까이 두는 법" 학습 → 출력이 **고정 길이 벡터(예: 1024-dim)**
> - 쓰임새: RAG 시스템의 **검색 단계**에서 질의·문서 벡터화 → cosine 유사도로 top-k 검색
> - 모델도 다름: BGE-M3 같은 전용 bi-encoder (decoder-only LLM이 아님)

### 3가지 핵심 개념

1. **Bi-Encoder**: 문장 하나당 벡터 하나를 독립적으로 생성 → 대규모 코퍼스에 미리 인코딩해두면 **cosine 한 번**으로 검색. Cross-Encoder와 달리 실시간 대규모 검색에 적합.
2. **MNR Loss (Multiple Negatives Ranking)**: positive pair만 학습 데이터로 넣고, **같은 배치 내 다른 샘플을 in-batch negative로 자동 활용**. label 작성 부담이 없고 배치 크기가 클수록 학습 신호가 강해짐.
3. **Matryoshka Loss**: 한 모델로 `[1024, 768, 512, 256, 128, 64]` **여러 차원에서 동시에 학습** → 배포 시 원하는 차원으로 truncate해도 성능 유지. 스토리지/지연시간에 따라 차원 선택 가능.

> LLM 메모리 해제 후 Embedding 모델 학습

In [ ]:
# Part 2 모델 메모리 해제
del model, trainer_tc
gc.collect(); torch.cuda.empty_cache()
print("✅ GPU 메모리 해제 완료")

# sentence-transformers: MNR/Matryoshka loss + 유사도 evaluator
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import MultipleNegativesRankingLoss, MatryoshkaLoss
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.similarity_functions import SimilarityFunction
from sentence_transformers.util import cos_sim
import numpy as np

### 3.1 데이터 합성 (RunPod에서 직접 생성)

외부 API 없이 **규칙 기반**으로 임베딩 학습 데이터를 합성합니다. ETF는 `카테고리·운용사·지수·설명` 같은 **구조화된 메타데이터**를 갖고 있어, 규칙만으로 고품질 쌍 생성이 가능합니다.

#### 쌍(pair) 생성 규칙과 학습 효과

| 쌍 종류 | 생성 규칙 | label | 학습 효과 |
|---|---|---|---|
| **Positive** (name ↔ desc) | 같은 ETF의 이름과 설명 | 1.0 | ETF 이름 ↔ 자연어 설명 매핑 |
| **Positive** (same category) | 같은 카테고리 ETF 2개 | 1.0 | 범주 clustering |
| **Hard Negative (후보)** | 같은 운용사, 다른 카테고리 | 0.0 | 운용사 공통이란 표면 특징에 속지 않기 |
| **Easy Negative** | 완전히 다른 카테고리 | 0.0 | 명확한 경계 학습 |
| **Query template** | "`{name}` ETF 정보" 등 쿼리 → 설명 | 1.0 | 검색 쿼리 패턴 학습 |

#### 💡 Hard Negative 개념

엄밀히 말해 Hard Negative의 "어려움"은 **임베딩 공간에서의 의미적 근접성**에서 나옵니다. '같은 운용사'는 표면 속성이므로 실제로 hard로 기능하는지는 학습 **후 cosine 유사도 측정으로 검증**해야 합니다 (실습 6 참고).

#### MNR + in-batch negative 원리

MNR Loss는 label이 없는 positive pair만 받고, **배치 안의 다른 positive 샘플들을 자동으로 negative로 간주**합니다. 배치 크기 32라면 1개의 positive vs 31개의 in-batch negative로 대조 학습 → 배치 크기가 클수록 강한 학습 신호.

In [ ]:
# Part 3: ETF 도메인 임베딩 학습 데이터 합성
#   Positive (1.0): 같은 의미 쌍    Hard Negative (0.0): 헷갈리는 쌍    Easy Negative (0.0): 명확히 다른 쌍
import pandas as pd, random
from itertools import combinations


# ETF 데이터: name/cat/mgr/idx/desc
# 설계: 카테고리별 3개↑, 같은 지수 복수 운용사, 인버스·레버리지(같은 운용사·반대 방향)로 Hard Negative 생성
etf_data = [
    # 국내주식 (대형): KOSPI200 추종 4종
    {"name": "KODEX 200",           "cat": "국내주식대형", "mgr": "삼성",     "idx": "KOSPI200",   "desc": "KOSPI 200 지수를 추종하는 국내 대표 대형주 ETF"},
    {"name": "TIGER 200",           "cat": "국내주식대형", "mgr": "미래에셋", "idx": "KOSPI200",   "desc": "KOSPI 200 지수 추종, 국내 시가총액 상위 200종목 투자"},
    {"name": "KBSTAR 200",          "cat": "국내주식대형", "mgr": "KB",       "idx": "KOSPI200",   "desc": "KOSPI 200 추종 저비용 대형주 ETF"},
    {"name": "ARIRANG 200",         "cat": "국내주식대형", "mgr": "한화",     "idx": "KOSPI200",   "desc": "KOSPI 200 추종 한화자산운용 대형주 ETF"},
    {"name": "KODEX 코스피",        "cat": "국내주식대형", "mgr": "삼성",     "idx": "KOSPI",      "desc": "KOSPI 전체 종목 추종하는 광범위 지수 ETF"},

    # 국내주식 (중소형·코스닥)
    {"name": "KODEX 코스닥150",     "cat": "국내주식중소", "mgr": "삼성",     "idx": "KOSDAQ150",  "desc": "코스닥 150 지수 추종, 중소형 성장주 투자"},
    {"name": "TIGER 코스닥150",     "cat": "국내주식중소", "mgr": "미래에셋", "idx": "KOSDAQ150",  "desc": "코스닥 시장 대표 중소형주 150종목 추종"},
    {"name": "KODEX 코스닥150 IT",  "cat": "국내주식중소", "mgr": "삼성",     "idx": "KOSDAQ150IT","desc": "코스닥 150 내 IT 업종 집중 투자 ETF"},

    # 해외주식 (미국): S&P500 3종 + NASDAQ100 2종
    {"name": "KODEX 미국S&P500",    "cat": "해외주식미국", "mgr": "삼성",     "idx": "S&P500",     "desc": "미국 S&P 500 지수 추종 대표 해외 ETF"},
    {"name": "TIGER 미국S&P500",    "cat": "해외주식미국", "mgr": "미래에셋", "idx": "S&P500",     "desc": "S&P 500 추종, 미국 대형주 500종목 분산 투자"},
    {"name": "ARIRANG 미국S&P500",  "cat": "해외주식미국", "mgr": "한화",     "idx": "S&P500",     "desc": "S&P 500 저비용 추종 한화자산운용 ETF"},
    {"name": "TIGER 나스닥100",     "cat": "해외주식미국", "mgr": "미래에셋", "idx": "NASDAQ100",  "desc": "나스닥 100 지수 추종, 미국 기술주 집중 ETF"},
    {"name": "KODEX 나스닥100",     "cat": "해외주식미국", "mgr": "삼성",     "idx": "NASDAQ100",  "desc": "나스닥 100 추종 미국 테크 대형주 ETF"},
    {"name": "KODEX 미국필라델피아반도체", "cat": "해외주식미국", "mgr": "삼성", "idx": "SOX",    "desc": "필라델피아 반도체 지수 추종, 미국 반도체 섹터 투자"},

    # 해외주식 (기타 지역): 지역 다양화
    {"name": "TIGER 차이나CSI300",  "cat": "해외주식기타", "mgr": "미래에셋", "idx": "CSI300",     "desc": "중국 CSI 300 지수 추종, 본토 대형주 투자"},
    {"name": "KODEX 일본TOPIX100",  "cat": "해외주식기타", "mgr": "삼성",     "idx": "TOPIX100",   "desc": "일본 TOPIX 100 지수 추종 대형주 ETF"},
    {"name": "TIGER 유로스탁스50",  "cat": "해외주식기타", "mgr": "미래에셋", "idx": "EuroStoxx50","desc": "유럽 대표 50종목 지수 EuroStoxx 50 추종"},
    {"name": "KODEX 인도Nifty50",   "cat": "해외주식기타", "mgr": "삼성",     "idx": "Nifty50",    "desc": "인도 Nifty 50 지수 추종 신흥국 ETF"},

    # 채권: 주식과 다른 자산군 → Easy Negative 소스
    {"name": "KODEX 국고채10년",    "cat": "채권",         "mgr": "삼성",     "idx": "KTB10Y",     "desc": "한국 국고채 10년물 장기 채권 투자"},
    {"name": "TIGER 국고채3년",     "cat": "채권",         "mgr": "미래에셋", "idx": "KTB3Y",      "desc": "국고채 3년물 중기 채권 투자 안정형"},
    {"name": "KODEX 단기채권",      "cat": "채권",         "mgr": "삼성",     "idx": "단기채",     "desc": "단기 채권 중심 초저위험 안정형 ETF"},
    {"name": "TIGER 미국채10년",    "cat": "채권",         "mgr": "미래에셋", "idx": "UST10Y",     "desc": "미국 국채 10년물 투자 달러 자산 ETF"},
    {"name": "KBSTAR 회사채",       "cat": "채권",         "mgr": "KB",       "idx": "회사채AA-",  "desc": "AA- 등급 우량 회사채 분산 투자"},

    # 인버스·레버리지: Hard Negative 핵심 소스 (같은 운용사·같은 지수·반대 방향)
    {"name": "KODEX 인버스",        "cat": "인버스",       "mgr": "삼성",     "idx": "KOSPI200",   "desc": "KOSPI 200 일일 수익률 역방향 -1배 추종"},
    {"name": "TIGER 인버스",        "cat": "인버스",       "mgr": "미래에셋", "idx": "KOSPI200",   "desc": "KOSPI 200 -1배 인버스 단기 하락 헤지용"},
    {"name": "KODEX 200선물인버스2X","cat": "인버스",      "mgr": "삼성",     "idx": "KOSPI200",   "desc": "KOSPI 200 -2배 더블 인버스 고위험 상품"},
    {"name": "KODEX 레버리지",      "cat": "레버리지",     "mgr": "삼성",     "idx": "KOSPI200",   "desc": "KOSPI 200 일일 수익률 2배 레버리지 ETF"},
    {"name": "TIGER 레버리지",      "cat": "레버리지",     "mgr": "미래에셋", "idx": "KOSPI200",   "desc": "KOSPI 200 2배 레버리지 단기 상승 베팅용"},
    {"name": "TIGER 미국나스닥100레버리지","cat":"레버리지","mgr": "미래에셋", "idx": "NASDAQ100",  "desc": "나스닥 100 2배 레버리지 미국 테크 공격형"},

    # 원자재·대체투자
    {"name": "KODEX 골드선물",      "cat": "원자재",       "mgr": "삼성",     "idx": "금선물",     "desc": "금 선물 투자 인플레이션 헤지 대체자산"},
    {"name": "TIGER 금은선물",      "cat": "원자재",       "mgr": "미래에셋", "idx": "금은선물",   "desc": "금·은 선물 복합 투자 귀금속 ETF"},
    {"name": "KODEX WTI원유선물",   "cat": "원자재",       "mgr": "삼성",     "idx": "WTI",        "desc": "서부텍사스산 원유(WTI) 선물 투자"},
    {"name": "TIGER 구리실물",      "cat": "원자재",       "mgr": "미래에셋", "idx": "구리",       "desc": "구리 실물 가격 추종 산업금속 ETF"},

    # 리츠·섹터
    {"name": "KODEX 리츠",          "cat": "리츠",         "mgr": "삼성",     "idx": "KRX리츠",    "desc": "국내 상장 부동산 리츠 분산 투자 ETF"},
    {"name": "TIGER 리츠부동산인프라","cat": "리츠",        "mgr": "미래에셋", "idx": "리츠인프라", "desc": "리츠 및 인프라 자산 결합 인컴형 ETF"},
    {"name": "KODEX 미국리츠(H)",   "cat": "리츠",         "mgr": "삼성",     "idx": "미국리츠",   "desc": "미국 리츠 환헤지형 해외 부동산 ETF"},
]

print(f"📦 ETF 데이터: {len(etf_data)}개")


# 데이터 분포 확인
from collections import Counter

print("\n카테고리 분포:")
for cat, n in Counter(e["cat"] for e in etf_data).most_common():
    print(f"  {cat:<15}: {n}개")

print("\n운용사 분포:")
for mgr, n in Counter(e["mgr"] for e in etf_data).most_common():
    print(f"  {mgr:<10}: {n}개")


def make_text(e, style=0):
    """ETF 정보 → 자연어 문장 (3가지 스타일로 다양화 → 문장 패턴 암기 방지)."""
    templates = [
        f"{e['name']}은(는) {e['mgr']}자산운용의 {e['cat']} 상품으로, {e['desc']}입니다.",
        f"{e['desc']}. {e['mgr']}에서 운용하는 {e['cat']} 카테고리 ETF {e['name']}.",
        f"{e['cat']} ETF {e['name']} — {e['idx']} 기반 상품. {e['desc']}.",
    ]
    return templates[style % len(templates)]


# 검색 쿼리 템플릿
query_templates = [
    "{name} ETF 정보",
    "{cat} 투자 상품 추천",
    "{mgr}자산운용 {cat} 상품",
    "{idx} 추종 ETF",
    "{name} 총보수 비교",
]


def synthesize_pairs(data):
    """ETF 데이터 → (sentence1, sentence2, label) 쌍 합성."""
    pairs = []
    stats = {}

    cats = {}
    for e in data:
        cats.setdefault(e["cat"], []).append(e)


    # Positive 1: name ↔ description (3가지 스타일)
    n = len(pairs)
    for e in data:
        for s in range(3):
            pairs.append({"sentence1": e["name"], "sentence2": make_text(e, s), "label": 1.0})
    stats["pos_name↔desc"] = len(pairs) - n


    # Positive 2: 같은 카테고리 (다른 스타일로 짝짓기 → 문장 구조 아닌 의미만 학습)
    n = len(pairs)
    for cat, items in cats.items():
        for a, b in combinations(items, 2):
            pairs.append({"sentence1": make_text(a, 0), "sentence2": make_text(b, 1), "label": 1.0})
    stats["pos_same_cat"] = len(pairs) - n


    # Positive 3: 쿼리 → ETF 설명 (실제 검색 시나리오)
    n = len(pairs)
    for e in data:
        for tmpl in query_templates:
            q = tmpl.format(**e)
            pairs.append({"sentence1": q, "sentence2": make_text(e), "label": 1.0})
    stats["pos_query↔desc"] = len(pairs) - n


    # Hard Negative: 같은 운용사·다른 카테고리 (표면은 비슷, 의미는 다름)
    n = len(pairs)
    for i, a in enumerate(data):
        for b in data[i+1:]:
            if a["mgr"] == b["mgr"] and a["cat"] != b["cat"]:
                pairs.append({"sentence1": make_text(a, 0), "sentence2": make_text(b, 0), "label": 0.0})
    stats["neg_hard"] = len(pairs) - n


    # Easy Negative: 다른 운용사·다른 카테고리 (명확히 다름, 비대칭 스타일)
    n = len(pairs)
    all_cats = list(cats.keys())
    for i, c1 in enumerate(all_cats):
        for c2 in all_cats[i+1:]:
            for a in cats[c1]:
                for b in cats[c2]:
                    if a["mgr"] != b["mgr"]:
                        pairs.append({"sentence1": make_text(a, 0), "sentence2": make_text(b, 1), "label": 0.0})
    stats["neg_easy"] = len(pairs) - n


    # Query Negative: 쿼리 → 관련 없는 카테고리 ETF (상위 2개 템플릿만 사용)
    n = len(pairs)
    for e in data:
        others = [x for x in data if x["cat"] != e["cat"]]
        for tmpl in query_templates[:2]:
            q = tmpl.format(**e)
            neg = random.choice(others)
            pairs.append({"sentence1": q, "sentence2": make_text(neg), "label": 0.0})
    stats["neg_query"] = len(pairs) - n

    return pairs, stats


random.seed(42)
pairs, stats = synthesize_pairs(etf_data)


# 중복 제거 ((정렬된 s1, s2) + label 기준)
seen, unique = set(), []
for p in pairs:
    key = tuple(sorted([p["sentence1"], p["sentence2"]])) + (p["label"],)
    if key in seen:
        continue
    seen.add(key)
    unique.append(p)

pairs = unique
random.shuffle(pairs)


# 결과 출력
print("📊 쌍 생성 내역:")
for k, v in stats.items():
    print(f"  {k:<20}: {v:>4}개")

pos = sum(1 for p in pairs if p["label"] == 1.0)
neg = len(pairs) - pos
print(f"\n✅ 합성 데이터: {len(pairs)}개 (중복 제거 후)")
print(f"   Positive: {pos} ({pos/len(pairs):.1%}), Negative: {neg} ({neg/len(pairs):.1%})")

In [ ]:
# Part 3 데이터 분할: MNR 학습셋 / Triplet 학습셋 / 평가셋
from collections import defaultdict


# Train/Test 분할 (9:1, 최소 평가셋 10개 확보)
random.shuffle(pairs)
n_test = max(10, len(pairs) // 10)
test_pairs  = pairs[:n_test]
train_pairs = pairs[n_test:]


# MNR 학습셋: Positive만 추출
# (배치 내 다른 쌍의 positive가 자동으로 negative 역할)
train_pos = [p for p in train_pairs if p["label"] == 1.0]
train_mnr = Dataset.from_list([
    {"sentence1": p["sentence1"], "sentence2": p["sentence2"]}
    for p in train_pos
])


# Triplet 학습셋 (참고용): anchor + positive + hard_negative 명시
# MNR의 in-batch negative보다 강한 학습 신호
pos_by_anchor = defaultdict(list)
neg_by_anchor = defaultdict(list)
for p in train_pairs:
    if p["label"] == 1.0:
        pos_by_anchor[p["sentence1"]].append(p["sentence2"])
    else:
        neg_by_anchor[p["sentence1"]].append(p["sentence2"])

triplets = []
for anchor, positives in pos_by_anchor.items():
    negs = neg_by_anchor.get(anchor, [])
    if not negs:
        continue
    for pos in positives:
        triplets.append({
            "anchor":   anchor,
            "positive": pos,
            "negative": random.choice(negs),
        })

train_triplet = Dataset.from_list(triplets) if triplets else None


# 평가셋: Positive/Negative 혼합 (분리 능력 측정용)
test_ds = Dataset.from_list(test_pairs)

test_pos = sum(1 for p in test_pairs if p["label"] == 1.0)
test_neg = len(test_pairs) - test_pos


print(f"📊 데이터 분할 결과:")
print(f"  MNR 학습 (pair):     {len(train_mnr):>4}개 (positive only)")
if train_triplet:
    print(f"  Triplet 학습:        {len(train_triplet):>4}개 (anchor+pos+hard_neg)")
print(f"  평가:                {len(test_ds):>4}개 (Positive {test_pos} / Negative {test_neg})")


# ⚠️ 학습 시 BatchSamplers.NO_DUPLICATES 사용 권장
# → 같은 anchor의 다른 positive가 같은 배치에 들어가 false negative가 되는 문제 방지
print(f"\n💡 학습 시 BatchSamplers.NO_DUPLICATES 사용 권장:")
print(f"   같은 anchor의 다른 positive가 같은 배치에 들어가 false negative가")
print(f"   되는 문제를 방지. 이번 데이터처럼 한 anchor에 여러 positive가")
print(f"   있는 구조(예: KODEX 200 ↔ 3가지 style의 설명문)에서 특히 중요.")

In [ ]:
# Part 3: BGE-M3 로드 + Loss(MNR+Matryoshka) + Evaluator + Before 측정


# 이전 모델 정리 (del + gc + empty_cache)
import gc, torch

try:
    del model, trainer_tc
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()


# BGE-M3 로드 (bf16)
#   다국어 임베딩 100+ 언어, 1024차원, Matryoshka 구조 지원
#   ⚠️ bf16은 Ampere 이후(A100/RTX3000+); T4·V100은 float16 사용
emb_model = SentenceTransformer(
    "BAAI/bge-m3",
    model_kwargs={"torch_dtype": torch.bfloat16},
)
print(f"✅ BGE-M3: {emb_model.get_sentence_embedding_dimension()}차원 (bf16)")
print(f"   VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


# Loss: MatryoshkaLoss(MNR) → 한 번의 학습으로 여러 차원 동시 학습
#   MNR: in-batch negative 대조 학습
#   Matryoshka (MRL, Google 2022): 앞 N차원만 잘라도 의미 유지
base_loss = MultipleNegativesRankingLoss(emb_model)

# dims 내림차순 필수 (matryoshka 내부 구현이 앞쪽 차원 잘라내는 방식)
dims = [1024, 512, 128]
train_loss = MatryoshkaLoss(emb_model, base_loss, matryoshka_dims=dims)


# Evaluator: 차원별 AP (BinaryClassification)
#   label이 0/1 이진값이므로 AP가 Spearman보다 적합
#   truncate_dim으로 앞 d차원만 평가 → Matryoshka 효과 확인
from sentence_transformers.evaluation import (
    BinaryClassificationEvaluator,
    SequentialEvaluator,
)

evaluators = [
    BinaryClassificationEvaluator(
        sentences1=test_ds["sentence1"],
        sentences2=test_ds["sentence2"],
        labels=[int(s) for s in test_ds["label"]],
        name=f"etf_dim{d}",
        truncate_dim=d,
    )
    for d in dims
]

# SequentialEvaluator: 여러 evaluator를 묶어 한 번에 실행
evaluator = SequentialEvaluator(evaluators)


# Before: 학습 전 baseline 측정
# 기대치: 범용 BGE-M3의 ETF 도메인 AP 0.6~0.7 → 학습 후 0.9+ 목표
before = evaluator(emb_model)

print("\n📊 Before 학습 (차원별 AP):")
for d in dims:
    # 버전별 key fallback
    ap = before.get(f"etf_dim{d}_cosine_ap", before.get(f"etf_dim{d}_ap"))
    print(f"  dim={d:>4}: AP={ap:.4f}")

In [ ]:
# Part 3 학습 (하이브리드): MNR + Triplet 병행 학습
#   MNR 데이터 → in-batch negative로 넓은 학습
#   Triplet 데이터 → 명시적 hard negative로 정교한 학습
from sentence_transformers.training_args import (
    BatchSamplers,
    MultiDatasetBatchSamplers,
)


# 데이터 준비 상태 확인
print("📦 학습 데이터 구성:")
print(f"  MNR (2열):     {len(train_mnr):>4}개 — positive only")
print(f"  Triplet (3열): {len(train_triplet):>4}개 — anchor+pos+hard_neg")
print(f"  비율:          {len(train_mnr)/len(train_triplet):.1f} : 1 (MNR/Triplet)")

print(f"\n🔍 데이터 샘플 확인:")
print(f"  MNR[0]:     {train_mnr[0]}")
print(f"  Triplet[0]: {train_triplet[0]}")


# Evaluator metric key 확인 (버전별 이름 차이 대비)
debug_result = evaluator(emb_model)
print("\n📋 사용 가능한 Evaluator metric keys:")
for k in list(debug_result.keys())[:5]:
    print(f"  {k}")
print("  ...")


# Trainer 구성 (dict 형태 → 멀티 데이터셋 모드 자동 전환)
emb_trainer = SentenceTransformerTrainer(
    model=emb_model,

    train_dataset={
        "mnr":     train_mnr,
        "triplet": train_triplet,
    },

    # 같은 MatryoshkaLoss 재사용 → 내부에서 2열/3열 포맷 자동 적응
    loss={
        "mnr":     train_loss,
        "triplet": train_loss,
    },

    evaluator=evaluator,

    args=SentenceTransformerTrainingArguments(
        output_dir=f"{WORK_DIR}/bge_m3_ft_hybrid",
        num_train_epochs=5,

        per_device_train_batch_size=32,

        learning_rate=2e-5,
        warmup_ratio=0.1,
        bf16=True,

        # 배치 샘플러:
        #   NO_DUPLICATES: 같은 anchor가 한 배치에 두 번 안 들어감 (false neg 방지)
        batch_sampler=BatchSamplers.NO_DUPLICATES,

        #   ROUND_ROBIN: 데이터셋 번갈아 샘플링 (비율 차이 큰 경우 권장)
        #   PROPORTIONAL: 크기 비례 샘플링
        multi_dataset_batch_sampler=MultiDatasetBatchSamplers.ROUND_ROBIN,

        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,

        metric_for_best_model="eval_etf_dim1024_cosine_ap",
        greater_is_better=True,

        logging_steps=5,
        report_to="none",
        seed=42,
    ),
)


# 학습 실행
print("\n🚀 하이브리드 학습 시작 (MNR + Triplet 병행)...")
emb_trainer.train()
print("✅ 학습 완료!")


# After 평가
after_hybrid = evaluator(emb_model)
print("\n📊 하이브리드 학습 결과 (차원별 AP):")
for d in dims:
    key = f"etf_dim{d}_cosine_ap"
    b = before.get(key, 0)
    a = after_hybrid.get(key, 0)
    print(f"  dim={d:>4}: Before={b:.4f} → After={a:.4f} ({a-b:+.4f})")

In [ ]:
# Part 3 평가: Spearman 기반 Before/After + Matryoshka 시각화
#   AP + Spearman 이중 검증 → 두 지표 동시 개선이면 견고한 학습
import gc, torch, numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator


# 방어 코드: 셀 재실행 시 변수 타입 꼬임 대비
if 'after' not in globals() or not isinstance(after, dict):
    print("⚠️ after가 dict가 아님 → 재측정")
    after = evaluator(emb_model)

if 'before' not in globals() or not isinstance(before, dict):
    print("⚠️ before가 dict가 아님 → baseline 재측정")
    _tmp_baseline = SentenceTransformer("BAAI/bge-m3", model_kwargs={"torch_dtype": torch.bfloat16})
    before = evaluator(_tmp_baseline)
    del _tmp_baseline
    gc.collect(); torch.cuda.empty_cache()


# Spearman 측정용 baseline 로드 (After는 이미 학습됨)
baseline = SentenceTransformer(
    "BAAI/bge-m3",
    model_kwargs={"torch_dtype": torch.bfloat16},
)


# 차원별 Spearman 측정 (Before & After)
before_dim_scores = {}
after_dim_scores  = {}

for d in dims:
    ev = EmbeddingSimilarityEvaluator(
        sentences1=test_ds["sentence1"],
        sentences2=test_ds["sentence2"],
        scores=[float(s) for s in test_ds["label"]],
        name=f"d{d}",
        truncate_dim=d,
    )

    for label, mdl, store in [("before", baseline, before_dim_scores),
                              ("after",  emb_model, after_dim_scores)]:
        result = ev(mdl)
        # 버전별 key 이름 fallback
        for k in (f"d{d}_spearman_cosine", f"d{d}_cosine_spearman"):
            if k in result:
                store[d] = result[k]
                break
        else:
            raise KeyError(
                f"Spearman key를 찾을 수 없음 ({label}, dim={d}). "
                f"반환된 keys: {list(result.keys())}"
            )

del baseline
gc.collect(); torch.cuda.empty_cache()


# 요약 표: AP + Spearman
print(f"\n{'Dim':>6} | {'AP B':>6} {'AP A':>6} {'ΔAP':>7} | {'Sp B':>6} {'Sp A':>6} {'ΔSp':>7}")
print("─" * 64)
for d in sorted(dims):
    ap_b = before.get(f"etf_dim{d}_cosine_ap", 0)
    ap_a = after.get(f"etf_dim{d}_cosine_ap",  0)
    sp_b = before_dim_scores[d]
    sp_a = after_dim_scores[d]
    print(f"{d:>6} | {ap_b:>6.3f} {ap_a:>6.3f} {ap_a-ap_b:>+7.3f} | "
          f"{sp_b:>6.3f} {sp_a:>6.3f} {sp_a-sp_b:>+7.3f}")


# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 왼쪽: 1024차원 Before/After bar
axes[0].bar(
    ['Before', 'After'],
    [before_dim_scores[1024], after_dim_scores[1024]],
    color=['#95a5a6', '#2ecc71'],
    width=0.4,
)
axes[0].set_ylim(0, 1)
axes[0].set_title('Full (1024-dim) Before vs After')
axes[0].set_ylabel('Spearman')

for i, s in enumerate([before_dim_scores[1024], after_dim_scores[1024]]):
    axes[0].text(i, s + 0.02, f'{s:.3f}', ha='center', fontweight='bold')


# 오른쪽: Matryoshka 차원별 곡선 + Before 1024 기준선
sorted_dims = sorted(dims)

axes[1].plot(
    sorted_dims,
    [before_dim_scores[d] for d in sorted_dims],
    'o--', color='#95a5a6', lw=2, label='Before',
)
axes[1].plot(
    sorted_dims,
    [after_dim_scores[d] for d in sorted_dims],
    'o-',  color='#e74c3c', lw=2, label='After',
)

axes[1].axhline(
    y=before_dim_scores[1024],
    color='#7f8c8d', linestyle=':', alpha=0.7,
    label=f'Before 1024 baseline ({before_dim_scores[1024]:.3f})',
)

axes[1].set_xscale('log', base=2)
axes[1].set_xticks(sorted_dims)
axes[1].set_xticklabels(sorted_dims)
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Spearman')
axes[1].set_title('Matryoshka: Score by Dimension')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 체크:
#   1. After(빨강)이 Before(회색)보다 전반적으로 위 → 학습 효과
#   2. After 기울기 완만 → Matryoshka 작동
#   3. After의 128차원이 Before 1024 기준선 위 → 8배 압축 가능

In [ ]:
# Part 3 마무리: 의미 기반 검색 데모 (RAG retriever 구조 동일)
import gc, torch, numpy as np


# Corpus: 전체 ETF를 자연어 설명문으로 벡터화
corpus = [make_text(e) for e in etf_data]


# Before/After 검색 비교
print("🔍 Before/After 검색 품질 비교\n")

baseline = SentenceTransformer("BAAI/bge-m3", model_kwargs={"torch_dtype": torch.bfloat16})

# 쿼리 4가지 성격: 의도 추론 / 명확 카테고리 / 위험 선호 / 간접 표현
queries = [
    "미국 기술주 투자하고 싶어",
    "안전한 채권 ETF",
    "KOSPI 2배 공격적",
    "인플레이션 대비 금 투자",
    "은퇴 후 안정적 수익",
    "반도체 섹터 집중",
]

for q in queries:
    print(f"Query: {q}")

    for label, mdl in [("Before", baseline), ("After ", emb_model)]:
        # normalize_embeddings=True → 내적이 곧 cosine 유사도
        c_emb = mdl.encode(corpus, convert_to_numpy=True, normalize_embeddings=True)
        q_emb = mdl.encode(q, normalize_embeddings=True)

        sims = c_emb @ q_emb

        top3 = np.argsort(-sims)[:3]

        result = " | ".join(f"{etf_data[i]['name']}({sims[i]:.2f})" for i in top3)
        print(f"  [{label}] {result}")

    print()

del baseline
gc.collect()
torch.cuda.empty_cache()


# Matryoshka 차원별 검색 비교: 128차원이 1024와 비슷하면 8배 저장/속도 이득
print("🎯 Matryoshka 차원별 검색 비교 (같은 쿼리):\n")

demo_query = "미국 기술주 투자하고 싶어"
print(f"Query: {demo_query}\n")

full_corpus_emb = emb_model.encode(corpus, convert_to_numpy=True, normalize_embeddings=True)
full_q_emb = emb_model.encode(demo_query, normalize_embeddings=True)

for d in [1024, 512, 128]:
    # 앞 d차원만 사용 후 재정규화 필수 (truncate하면 길이 1 아님)
    c_emb_d = full_corpus_emb[:, :d]
    q_emb_d = full_q_emb[:d]

    c_emb_d = c_emb_d / np.linalg.norm(c_emb_d, axis=1, keepdims=True)
    q_emb_d = q_emb_d / np.linalg.norm(q_emb_d)

    sims = c_emb_d @ q_emb_d
    top3 = np.argsort(-sims)[:3]
    result = " | ".join(f"{etf_data[i]['name']}({sims[i]:.2f})" for i in top3)
    print(f"  dim={d:>4}: {result}")

print("\n💡 dim=1024 / 512 / 128 결과가 거의 같다면 Matryoshka 성공 →")
print("   프로덕션에선 128차원으로 배포하여 저장·속도 8배 이득")

In [ ]:
# 로컬 저장 (전체 모델 ~2GB, Matryoshka 차원 정보 포함)
emb_model.save(f"{WORK_DIR}/bge_m3_finetuned")
print(f"✅ BGE-M3 로컬 저장: {WORK_DIR}/bge_m3_finetuned")

# HF Hub 업로드
HF_USERNAME = "redwiggler"                      # ✏️ 본인 HF 계정명
REPO_NAME   = "bge-m3-etf-matryoshka"           # ✏️ 리포 이름
HUB_REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

emb_model.push_to_hub(
    HUB_REPO_ID,
    token=os.environ.get("HF_TOKEN"),
    private=True,                               # ✏️ 공개하려면 False
    exist_ok=True,
)
print(f"✅ Hub 업로드 완료: https://huggingface.co/{HUB_REPO_ID}")

---
## 실습 문제

### 실습 1: SFT 데이터 증강
KoAlpaca에서 특정 도메인(예: 금융)만 필터링하여 SFT를 수행하고, 범용 학습과 ROUGE를 비교하세요.

### 실습 2: Tool 추가
`search_product`, `calculate_expense_ratio` 등 커스텀 Tool을 정의하고 학습된 모델이 올바르게 호출하는지 테스트하세요. (`temperature=0.1`)

### 실습 3: 임베딩 데이터 확장
`etf_data`에 10개 이상의 ETF를 추가하고, 합성 데이터를 늘려 Before/After를 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요.

---

### 참고 자료
- Unsloth: https://github.com/unslothai/unsloth
- kiwipiepy: https://github.com/bab2min/kiwipiepy
- BGE-M3: https://huggingface.co/BAAI/bge-m3
- glaive: https://huggingface.co/datasets/glaiveai/glaive-function-calling-v2
- Matryoshka: Kusupati et al., NeurIPS 2022